# Brain clot (haemorrhage) segmentation on head CT: 2D, runs on a free Colab T4

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells in order.
Results are saved to Google Drive. Research use only, not for diagnosis.

## 1. GPU

In [ ]:
!nvidia-smi -L

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/brain-clot"
OUT = f"{BASE}/models/clot"
os.makedirs(OUT, exist_ok=True); os.makedirs(f"{BASE}/downloads", exist_ok=True)
print("Results ->", OUT)

## 3. Get the code
Uses the `GITHUB_TOKEN` Colab secret (read-only) while the repo is private.

In [ ]:
import subprocess, shutil
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/ChanduVenkatSaiNarne/Brain-Clot-Edema-midlineshift.git" if token else "https://github.com/ChanduVenkatSaiNarne/Brain-Clot-Edema-midlineshift.git"
REPO = "/content/Brain-Clot-Edema-midlineshift"
if not os.path.exists(f"{REPO}/.git"):
    shutil.rmtree(REPO, ignore_errors=True)
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    print((r.stderr.replace(token, "***") if token else r.stderr) or "Cloned.")
%cd /content/Brain-Clot-Edema-midlineshift
!git fetch -q && git checkout -q master && git reset -q --hard origin/master
!git log --oneline -1
assert os.path.exists("train_2d.py"), "Code download failed"
!pip -q install -r requirements.txt

## 4. Get BHSD (192 labelled CT scans; downloaded once and kept in Drive)

In [ ]:
ZIP = f"{BASE}/downloads/bhsd_label_192.zip"
if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1e6:
    !wget -q --show-progress -O "{ZIP}" "https://huggingface.co/datasets/WuBiao/BHSD/resolve/main/label_192.zip"
SRC = "/content/data/bhsd"
!mkdir -p {SRC} && unzip -q -o "{ZIP}" -d {SRC}
!find {SRC} -maxdepth 3 -type d; echo; find {SRC} -type f | wc -l

## 5. Cut the scans into 2D slices (3 CT windows each)

In [ ]:
DATA = "/content/data/bhsd_256"
!python prepare_bhsd.py --src {SRC} --out {DATA} --size 256

## 6. Quick test (2 epochs)

In [ ]:
!python train_2d.py --data {DATA} --out /content/test_run --epochs 2

## 7. Full training (about 1.5–2.5 h on a T4; keep the tab open)

In [ ]:
!python train_2d.py --data {DATA} --out {OUT}

## 8. Test results and curves

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
r = json.load(open(f"{OUT}/test_results.json"))
print({k: v for k, v in r.items() if k not in ("per_scan", "test_scans")})
h = pd.read_csv(f"{OUT}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h.epoch, h.loss); ax[0].set_title("Training loss")
ax[1].plot(h.epoch, h.val_dice_3d); ax[1].set_title("Validation 3D Dice"); plt.show()

## 8b. Tune threshold on validation scans (no retraining, about 5 minutes)
Picks the threshold and speck size on the validation scans, then reports the test scans once, plus Dice by bleed size and the worst scans.

In [ ]:
!python tune_clot.py --data {DATA} --model {OUT}

## 9. Predict a held-out test scan (red = model, green = expert)

In [ ]:
from data_ct import find_pairs, stem
from IPython.display import Image, display
pairs, _, _ = find_pairs(SRC)
by_id = {stem(i).replace(" ", "_"): (i, l) for i, l in pairs}
case = r["test_scans"][0]
ct, truth = by_id[case]
!python predict.py --ckpt {OUT}/clot_best.pt --ct "{ct}" --truth "{truth}" --out {BASE}/predictions
display(Image(f"{BASE}/predictions/{case}_overlay.png", width=1000))

## 10. Experiment: crop to the head first (about 1 h on an L4)
Same model and settings; only the preprocessing changes. Uses the same test scans, so the result compares directly with section 7.

In [ ]:
DATA_CROP = "/content/data/bhsd_256_crop"
OUT_CROP = f"{BASE}/models/clot_crop"
!python prepare_bhsd.py --src {SRC} --out {DATA_CROP} --size 256 --crop

In [ ]:
!python train_2d.py --data {DATA_CROP} --out {OUT_CROP}

In [ ]:
!python tune_clot.py --data {DATA_CROP} --model {OUT_CROP}